# Mean & RMS Analysis (Figure 7)

Reproduction of **Figure 7** of Nagatani (2006): the mean values and the
root-mean-square (RMS) fluctuation of the **headways** $H_1,H_2$ and the
**tour times** $\Delta T_1,\Delta T_2$ of the two buses, plotted against the
loading parameter $\Gamma$ for the speedup case $S_1=0.5,\ S_2=0.2$.

- **Headway** $H_i(m)$: gap between bus $i$'s arrival and the arrival of the bus
  right before it (buses pass each other freely, so "the bus before" is decided
  by arrival order).
- **Tour time** $\Delta T_i(m)=T_i(m{+}1)-T_i(m)$: one full cycle of bus $i$.
- **Mean**: average over the steady-state window; **RMS**: $\sqrt{\langle(x-\langle x\rangle)^2\rangle}$,
  i.e. the standard deviation, measuring how irregular the schedule is.

Statistics use the steady-state trips $1000\le m\le 2000$.


## Setup

We reuse the project's `simulator` package instead of re-implementing the map. Two
pieces are imported:

- `simulate_bus_system` — iterates the dimensionless recurrence
  $T_i(m{+}1)=T_i(m)+\Gamma\,(T_i(m)-T_{i'}(m'))+\dfrac{1}{1+S_i\,(T_i(m)-T_{i'}(m'))}$
  for two buses that pass each other freely.
- `generate_sweep_csv` — runs that simulation across many $\Gamma$ values and writes the
  per-$\Gamma$ summary statistics to a CSV (the data layer for this figure).

The notebook lives in `src/model_analysis/`, so we add `src/` to `sys.path` to make the
`simulator` package importable from here.


In [ ]:
import sys, os

SRC = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC not in sys.path:
    sys.path.append(SRC)

import numpy as np
import matplotlib.pyplot as plt
from simulator.data_export import generate_sweep_csv


## Data

For each loading parameter $\Gamma$ we run the simulation and, on the **steady-state**
window of trips $1000\le m\le 2000$ (early trips are discarded as transient), we collect:

- the **headways** $H_1,H_2$ — for each arrival, the gap to the previous arrival in the
  interleaved chronological order (because the buses overtake each other, "the bus before"
  is whichever arrived last, not a fixed bus);
- the **tour times** $\Delta T_1,\Delta T_2$ — the gap between two successive arrivals of
  the *same* bus.

`generate_sweep_csv` then stores, per $\Gamma$, the **mean** and the **standard deviation**
of each of these four quantities. Note that NumPy's `std` (population, `ddof=0`) is exactly
the RMS of the fluctuation, $\sqrt{\langle (x-\langle x\rangle)^2\rangle}$, which is what
Figure 7 plots. We use $S_1=0.5,\ S_2=0.2$ (the same speedup case as Fig. 2(d) / 3(d)),
sweep $600$ values of $\Gamma\in(0,2]$, and read the CSV back into arrays for plotting.


In [ ]:
CSV = os.path.join(SRC, "model_analysis", "data", "sweep_S05_S02.csv")

generate_sweep_csv(
    output_path=CSV,
    T1_initial=1.0, T2_initial=2.5,
    S1=0.5, S2=0.2,
    num_trips=2000,
    gamma_range=(1e-3, 2.0),
    num_gamma=600,
    trip_min=1000, trip_max=2000,
)

import csv
rows = list(csv.reader(open(CSV)))
hdr, raw = rows[0], np.array(rows[1:], dtype=float)
col = {name: raw[:, k] for k, name in enumerate(hdr)}

G = col["gamma"]
H1a, H2a = col["h1_mean"], col["h2_mean"]
T1a, T2a = col["t1_mean"], col["t2_mean"]
H1v, H2v = col["h1_std"], col["h2_std"]
T1v, T2v = col["t1_std"], col["t2_std"]


## Figure 7

The four panels plot, against $\Gamma$, the **mean** values (top row) and the **RMS**
fluctuation (bottom row) of the headways and tour times; panels (b) and (d) zoom into
$0<\Gamma<0.5$ where the transitions happen.

Colour / line conventions:

- **red** = headways $H$, **blue** = tour times $\Delta T$;
- **solid** = bus 1, **dashed** = bus 2;
- the dotted vertical lines mark the transition points of Fig. 3(d): point 1
  ($\Gamma=0.167$, regular $\to$ periodic), point 2 ($\Gamma=0.248$, periodic $\to$ chaos),
  point 3 ($\Gamma=0.407$).

Reading the panels together shows how both the average schedule (means) and its
irregularity (RMS) evolve across the regular, periodic and chaotic regimes.


In [ ]:
pts = [("1", 0.167), ("2", 0.248), ("3", 0.407)]

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

def style(a, xlim, ylim, ylabel, title, zoom=False):
    a.set_xlim(xlim); a.set_ylim(ylim)
    a.set_xlabel(r"$\Gamma$ (loading parameter)"); a.set_ylabel(ylabel)
    a.set_title(title); a.legend(fontsize=9, loc="upper left"); a.grid(alpha=.25)
    if zoom:
        for lab, gp in pts:
            a.axvline(gp, color="0.5", ls=":", lw=.9)
            a.annotate(lab, (gp, ylim[1]), textcoords="offset points",
                       xytext=(2, -12), fontsize=8, color="0.35")

for a, xl, yl, t, z in [(ax[0, 0], (0, 2), (0, 3), "(a) Mean values, $0<\\Gamma<2$", False),
                        (ax[0, 1], (0, 0.5), (0, 1.4), "(b) Enlargement, $0<\\Gamma<0.5$", True)]:
    a.plot(G, T1a, "b-", lw=1.3, label=r"$\Delta T_{1a}$")
    a.plot(G, T2a, "b--", lw=1.3, label=r"$\Delta T_{2a}$")
    a.plot(G, H1a, "r-", lw=1.3, label=r"$H_{1a}$")
    a.plot(G, H2a, "r--", lw=1.3, label=r"$H_{2a}$")
    style(a, xl, yl, "mean", t, z)

for a, xl, yl, t, z in [(ax[1, 0], (0, 2), (0, 1.4), "(c) RMS, $0<\\Gamma<2$", False),
                        (ax[1, 1], (0, 0.5), (0, 0.45), "(d) Enlargement, $0<\\Gamma<0.5$", True)]:
    a.plot(G, T1v, "b-", lw=1.3, label=r"$\Delta T_{1v}$")
    a.plot(G, T2v, "b--", lw=1.3, label=r"$\Delta T_{2v}$")
    a.plot(G, H1v, "r-", lw=1.3, label=r"$H_{1v}$")
    a.plot(G, H2v, "r--", lw=1.3, label=r"$H_{2v}$")
    style(a, xl, yl, "rms", t, z)

fig.suptitle(r"Figure 7 - Mean & RMS of headways and tour times ($S_1=0.5,\ S_2=0.2$)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(os.path.join(SRC, "model_analysis", "figure7_mean_rms.png"), dpi=140)
plt.show()


### Discussion

- **Regular region ($\Gamma<0.167$).** Both buses run at constant speed, so the **RMS of
  every quantity is zero** (panels c, d) — the schedule is perfectly predictable. The two
  buses share the *same* tour time ($\Delta T_{1a}=\Delta T_{2a}$) but have *different*
  headways; $H_{1a}\to 0$ and $H_{2a}\to$ tour time as $\Gamma\to$ point 1.
- **Point 1.** Regular $\to$ periodic: the RMS curves jump off zero, the tour times split.
- **Periodic / chaotic ($0.167<\Gamma<0.407$).** $H_{2v}$ grows monotonically; $H_{1v}$ dips
  until $\Gamma\approx0.333$ then rises; the headway RMS values cross near point 2 (onset of
  chaos). Tour-time RMS stays small until point 3.
- **Strongly chaotic ($\Gamma>0.407$) up to $\Gamma=2$.** Tour-time RMS rises monotonically;
  **all means and RMS diverge at $\Gamma=2$** — the schedule breaks down.

**Conclusion.** The mean values describe the average schedule and the RMS values measure its
irregularity: zero when regular, growing through the periodic and chaotic regimes, diverging
at $\Gamma=2$. This is the quantitative signature of schedule instability that the speedup
control is designed to suppress.
